In [1]:
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.strategies import DDPStrategy
from loaders.Sdataloader import Stanford_Dataset
from model.GNN_inf_seg import Lightning_GNN
from sklearn.model_selection import train_test_split
import torch_geometric as tg
import numpy as np
import torch
import lightning as pl
import datetime
import yaml
import os
import wandb
import itertools

In [2]:
with open('configs/config_S3DIS.yml', 'r') as f:
    config = yaml.safe_load(f)
dataset_test = Stanford_Dataset(root=config['root'],
                                split='test')

test_loader = tg.loader.DataLoader(dataset_test,
                                    batch_size=config['batch_size'],
                                    num_workers=2,
                                    shuffle=False,)

In [9]:
GNN_model = Lightning_GNN(config=config)
GNN_model.load_state_dict(torch.load('/home/lars/models/2024-09-15_18.03.45s3dis_base/epoch=599-train_loss=0.06.ckpt')['state_dict'])
GNN_model.to('cpu')
a = 1

In [3]:
# get sample
number = 11
dataloader_iter = iter(test_loader)
sample = next(itertools.islice(dataloader_iter, number-1, None))

In [11]:
with torch.no_grad():
    GNN_model.eval()
    out_pc = GNN_model(sample)

prediction = torch.argmax(out_pc, dim=1)

In [12]:
# extract sample with lowest accuracy   
accr_list = [] 

for i in range(config['batch_size']):
    accr = torch.sum(prediction[sample.batch == i] == sample.y[sample.batch == i]).item() / len(sample.y[sample.batch == i])
    accr_list.append((accr, i))

lowest_accuracy_indices = [x[1] for x in sorted(accr_list, key=lambda x: x[0])[:3]]
print(lowest_accuracy_indices)

[1, 0]


In [13]:
accr_list

[(0.9409320738447067, 0), (0.9103205840819638, 1)]

In [4]:
batch_idx = 0
ax = 0

In [15]:
pos = sample.pos[sample.batch == batch_idx]
pos = pos - pos.mean(dim=0)
color = sample.x[sample.batch == batch_idx][:,:3][pos[:,ax] < 0]
label = sample.y[sample.batch == batch_idx][pos[:,ax] < 0]
pred = prediction[sample.batch == batch_idx][pos[:,ax] < 0]
pos = pos[pos[:,ax] < 0]

In [5]:
out_name = 'cloud_3'

In [17]:
# Save all pointclouds
os.makedirs('S3DIS_out/' + out_name,exist_ok=True)  
np.savetxt('S3DIS_out/'+out_name+'/input.txt', np.concatenate((pos.numpy(),color.numpy()), axis=1))
np.savetxt('S3DIS_out/'+out_name+'/label.txt', np.concatenate((pos.numpy(),label.unsqueeze(1).numpy()),axis=1))
np.savetxt('S3DIS_out/'+out_name+'/pred.txt', np.concatenate((pos.numpy(),pred.unsqueeze(1).numpy()),axis=1))           


In [19]:
# delete all outputs
del pos
del color
del label
del pred
del out_pc
del GNN_model

In [5]:
del GNN_model

In [6]:
with open('configs/config_S3DIS_global.yml', 'r') as f:
    config = yaml.safe_load(f)

GNN_model = Lightning_GNN(config=config)
GNN_model.load_state_dict(torch.load('/home/lars/models/2024-09-17_19.20.45s3dis_global/epoch=609-train_loss=0.08.ckpt')['state_dict'])

glob_sampel = tg.data.Data(
    x=sample.x[sample.batch == batch_idx],
    pos=sample.pos[sample.batch == batch_idx],
    batch=torch.zeros(sample.x[sample.batch == batch_idx].shape[0], dtype=torch.long),
    y=sample.y[sample.batch == batch_idx]
)
out_pc = GNN_model(glob_sampel)


: 

In [8]:
accr = torch.sum(torch.argmax(out_pc, dim=1) == sample.y[sample.batch == batch_idx]).item() / len(sample.y[sample.batch == batch_idx])
print(accr)

0.9581079283433842


In [7]:
pos = glob_sampel.pos
pos = pos - pos.mean(dim=0)
preds = torch.argmax(out_pc, dim=1)[pos[:,ax] < 0]
pos = pos[pos[:,ax] < 0]
np.savetxt('S3DIS_out/'+out_name+'/pred_global.txt', np.concatenate((pos.numpy(),preds.unsqueeze(1).numpy()),axis=1))           